# Chapter 23
## Entrainment by Excitatory Input Pulses
- Code by : [Abolfazl Ziaeemehr](https://github.com/Ziaeemehr)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ITNG/ModelingNeuralDynamics/blob/main/python/chapter23.ipynb)

## About this chapter

Periodic excitatory pulses can make a neuron's spikes adopt the drive's
timing. `simulate_lif_entrainment` builds an LIF trace by hand from its
closed-form exponential decay between pulses. `plot_f_entrainment`/
`plot_f_entrainment_2` plot the corresponding phase return map
$\alpha_{k+1}=F(\alpha_k)$ and a fixed-point iteration on it directly,
without simulating a neuron. The `simulate_wb_entrainment*` examples drive
a WB neuron with a periodic synaptic pulse train and read off its
pulse-relative spike phases: `simulate_wb_neuron_entrained` shows a
1-to-1 locked response, `simulate_wb_neuron_n_to_one` an n-to-1 pattern,
`simulate_wb_neuron_irregular` a non-locking response, and
`simulate_wb_entrainment_intervals` sweeps the synaptic strength to map
out where locking occurs -- its inner loop is JIT-compiled with numba,
since the uncompiled 201-value sweep (each ~10^6 steps) is, in the book's
own words, one that "takes quite a while to run".

See [`README.md`](chapter23.md) for the full guide, including suggested
order and related chapters.

In [ ]:
import subprocess
import sys
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "modelingneuraldynamics"], check=True)

In [ ]:
import math
import numpy as np
from numpy import exp
import matplotlib.pyplot as plt
from ipywidgets import interact
from numba import njit
from mnd.core import draw_arrow

## LIF Entrainment

An LIF neuron driven by periodic excitatory pulses of size $\epsilon$,
built directly from the closed-form exponential decay between pulses
($v\to v e^{-T/\tau}$) plus the instantaneous $+\epsilon$ jump -- no
numerical integration needed.

In [ ]:
def simulate_lif_entrainment(T=20.0, tau=30.0, epsilon=0.4875, t_final=1000.0):
    r = np.exp(-T / tau)
    num_pulses = round(t_final / T)

    segments = []  # (t_offset, v_post) for each inter-pulse decay curve
    reset_lines = []  # t where v hits threshold and resets to 0
    dotted_lines = []  # (t, v_pre, v_post) jump at each pulse

    v_post = epsilon
    for k in range(1, num_pulses + 1):
        segments.append((k * T, v_post))
        v_pre = v_post * r
        v_post = v_pre + epsilon
        v_post = v_post * (v_post < 1)
        if v_post == 0:
            reset_lines.append((k + 1) * T)
        dotted_lines.append(((k + 1) * T, v_pre, v_post))

    return segments, reset_lines, dotted_lines


def plot_lif_entrainment(segments, reset_lines, dotted_lines, T=20.0, tau=30.0, t_final=1000.0):
    t = np.arange(101) / 100 * T
    plt.figure(figsize=(7, 4))
    plt.plot([0, T], [0, 0], '-k', linewidth=4)
    plt.plot([T, T], [0, segments[0][1]], ':k', linewidth=1)

    for t_offset, v in segments:
        plt.plot(t_offset + t, v * np.exp(-t / tau), '-k', linewidth=2)
    for tt in reset_lines:
        plt.plot([tt, tt], [0, 5], '-k', linewidth=2)
    for tt, v_pre, v_post in dotted_lines:
        plt.plot([tt, tt], [v_pre, v_post], ':k', linewidth=1)

    plt.xlim(0, t_final)
    plt.ylim(0, 6)
    plt.xlabel('$t$ [ms]')
    plt.ylabel('$v$')
    plt.tight_layout()
    plt.show()

In [ ]:
plot_lif_entrainment(*simulate_lif_entrainment())

## Entrainment Phase Return Map

The map $F(\alpha)=(\alpha+\epsilon)e^{-T/\tau}$ taking the phase after
one pulse to the phase after the next, plotted against the identity line
-- a stable fixed point predicts a repeatable locked phase.

In [ ]:
def F_entrainment(alpha, T=25.0, tau=20.0, epsilon=0.2):
    return (alpha + epsilon) * np.exp(-T / tau)


def plot_f_entrainment(T=25.0, tau=20.0, epsilon=0.2):
    plt.figure(figsize=(6, 6))
    plt.plot([0, 1 - epsilon], [F_entrainment(0, T, tau, epsilon), F_entrainment(1 - epsilon, T, tau, epsilon)],
             '-k', linewidth=4)
    plt.plot([1 - epsilon, 1], [0, 0], '-k', linewidth=6)
    plt.plot([1 - epsilon, 1 - epsilon], [0, F_entrainment(1 - epsilon, T, tau, epsilon)], ':k', linewidth=1)
    plt.plot([0, 1], [0, 1], ':k', linewidth=2)

    plt.xlim(0, 1)
    plt.ylim(0, 1)
    plt.gca().set_box_aspect(1)
    plt.xticks([0, 1])
    plt.xlabel(r'$\alpha$')
    plt.ylabel('$F$')
    plt.tight_layout()
    plt.show()

In [ ]:
plot_f_entrainment()

## Fixed-Point Iteration on the Return Map

Iterates $F$ from $\alpha_1=0$ and traces the staircase path
$\alpha_1\to\alpha_2\to\alpha_3$ converging toward the stable fixed
point.

In [ ]:
def simulate_f_entrainment_2(T=25.0, tau=50.0, epsilon=0.6):
    alpha = [0.0]
    alpha.append(F_entrainment(alpha[0], T, tau, epsilon))
    alpha.append(F_entrainment(alpha[1], T, tau, epsilon))
    return alpha


def plot_f_entrainment_2(alpha, T=25.0, tau=50.0, epsilon=0.6):
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot([0, 1 - epsilon], [F_entrainment(0, T, tau, epsilon), F_entrainment(1 - epsilon, T, tau, epsilon)],
            '-k', linewidth=4)
    ax.plot([1 - epsilon, 1], [0, 0], '-k', linewidth=8)
    ax.plot([0, 1], [0, 1], ':k', linewidth=2)

    a1, a2, a3 = alpha
    ax.plot([0, 0], [0, a2], '-r', linewidth=2)
    ax.plot([0, a2], [a2, a2], '-r', linewidth=1)
    ax.plot([a2, a2], [a2, a3], '-r', linewidth=1)
    ax.plot([a2, a3], [a3, a3], '-r', linewidth=1)
    ax.plot([a3, a3], [a3, 0], '-r', linewidth=1)
    ax.plot([0, a3], [0, 0], '-r', linewidth=2)
    ax.plot([a2, a2], [0, a2], ':r', linewidth=1)
    ax.plot(a1, 0, '.r', markersize=20)
    ax.plot(a2, 0, '.r', markersize=20)
    ax.plot(a3, 0, '.r', markersize=20)

    xlim, ylim = (0, 1), (0, 1)
    arrows = [
        (0., a2 / 2, (0., 1.)),
        (a2 / 2, a2, (1., 0.)),
        (a2, a2 + (a3 - a2) / 2, (0., 1.)),
        ((a2 + a3) / 2, a3, (1., 0.)),
        (a3, a3 / 2, (0., -1.)),
        (a3 * 0.4, 0., (-1., 0.)),
    ]
    for x, y, v in arrows:
        draw_arrow(ax, xlim, ylim, x, y, np.array(v), epsilon=0.03, width=2, color='r')

    ax.text(-0.02, -0.08, r'$\alpha_1$', fontsize=18, color='r')
    ax.text(a2 - 0.02, -0.08, r'$\alpha_2$', fontsize=18, color='r')
    ax.text(a3 - 0.02, -0.08, r'$\alpha_3$', fontsize=18, color='r')

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_box_aspect(1)
    ax.set_xticks([1])
    ax.set_yticks([0, 0.5, 1])
    ax.set_ylabel('$F$')
    plt.tight_layout()
    plt.show()

In [ ]:
plot_f_entrainment_2(simulate_f_entrainment_2())

## WB Neuron under Periodic Excitatory Drive (shared by the examples below)

A WB neuron driven by a periodically-reset release variable $q$ (reset to
1 every $T$ ms) feeding the usual two-state synaptic gate $s$. Two
numba-jitted steppers share the same physics: one returns the full
voltage trace (for the trajectory plots), the other only the spike times
in the second half of the run (cheaper for the parameter sweep below).

In [ ]:
def tau_peak_function(tau_d, tau_r, tau_d_q):
    """Time (from a delta-function pulse of transmitter release) at which
    the synaptic gate s peaks."""
    dt = 0.01
    dt05 = dt / 2
    s, t = 0.0, 0.0
    s_inc = exp(-t / tau_d_q) * (1 - s) / tau_r - s * tau_d
    while s_inc > 0:
        t_old, s_inc_old = t, s_inc
        s_tmp = s + dt05 * s_inc
        s_inc_tmp = exp(-(t + dt05) / tau_d_q) * (1 - s_tmp) / tau_r - s_tmp / tau_d
        s = s + dt * s_inc_tmp
        t = t + dt
        s_inc = exp(-t / tau_d_q) * (1 - s) / tau_r - s / tau_d
    return (t_old * (-s_inc) + t * s_inc_old) / (s_inc_old - s_inc)


def tau_d_q_function(tau_d, tau_r, tau_hat):
    """Release time constant tau_d_q so that tau_peak_function reproduces
    the prescribed tau_hat (bisection, since there's no closed form)."""
    tau_d_q_left = 1.0
    while tau_peak_function(tau_d, tau_r, tau_d_q_left) > tau_hat:
        tau_d_q_left /= 2
    tau_d_q_right = tau_r
    while tau_peak_function(tau_d, tau_r, tau_d_q_right) < tau_hat:
        tau_d_q_right *= 2
    while tau_d_q_right - tau_d_q_left > 1e-12:
        tau_d_q_mid = (tau_d_q_left + tau_d_q_right) / 2
        if tau_peak_function(tau_d, tau_r, tau_d_q_mid) <= tau_hat:
            tau_d_q_left = tau_d_q_mid
        else:
            tau_d_q_right = tau_d_q_mid
    return (tau_d_q_left + tau_d_q_right) / 2


@njit
def _wb_entrainment_trace(g_syn, T, tau_d, tau_r, tau_dq, dt, dt05, m_steps,
                           c, g_k, g_na, g_l, v_k, v_na, v_l, i_ext):
    v = np.zeros(m_steps + 1)
    m = np.zeros(m_steps + 1)
    h = np.zeros(m_steps + 1)
    n = np.zeros(m_steps + 1)
    q = np.zeros(m_steps + 1)
    s = np.zeros(m_steps + 1)
    v[0] = -65.0
    alpha_m0 = 0.1 * (v[0] + 35) / (1 - math.exp(-(v[0] + 35) / 10))
    beta_m0 = 4 * math.exp(-(v[0] + 60) / 18)
    m[0] = alpha_m0 / (alpha_m0 + beta_m0)
    alpha_h0 = 0.35 * math.exp(-(v[0] + 58) / 20)
    beta_h0 = 5.0 / (math.exp(-0.1 * (v[0] + 28)) + 1)
    h[0] = alpha_h0 / (alpha_h0 + beta_h0)
    alpha_n0 = 0.05 * (v[0] + 34) / (1 - math.exp(-0.1 * (v[0] + 34)))
    beta_n0 = 0.625 * math.exp(-(v[0] + 44) / 80)
    n[0] = alpha_n0 / (alpha_n0 + beta_n0)

    for k in range(m_steps):
        t = k * dt
        if k > 0 and abs(round(t / T) - t / T) < 1e-12:
            q[k] = 1.0

        alpha_h_v = 0.35 * math.exp(-(v[k] + 58) / 20)
        alpha_n_v = 0.05 * (v[k] + 34) / (1 - math.exp(-0.1 * (v[k] + 34)))
        beta_h_v = 5.0 / (math.exp(-0.1 * (v[k] + 28)) + 1)
        beta_n_v = 0.625 * math.exp(-(v[k] + 44) / 80)

        v_inc = (g_k * math.pow(n[k], 4.0) * (v_k - v[k]) + g_na * math.pow(m[k], 3.0) * h[k] * (v_na - v[k])
                 + g_l * (v_l - v[k]) - g_syn * s[k] * v[k] + i_ext) / c
        h_inc = alpha_h_v * (1 - h[k]) - beta_h_v * h[k]
        n_inc = alpha_n_v * (1 - n[k]) - beta_n_v * n[k]
        s_inc = q[k] * (1 - s[k]) / tau_r - s[k] / tau_d
        q_inc = -q[k] / tau_dq

        v_tmp = v[k] + dt05 * v_inc
        alpha_m_vtmp = 0.1 * (v_tmp + 35) / (1 - math.exp(-(v_tmp + 35) / 10))
        beta_m_vtmp = 4 * math.exp(-(v_tmp + 60) / 18)
        m_tmp = alpha_m_vtmp / (alpha_m_vtmp + beta_m_vtmp)
        h_tmp = h[k] + dt05 * h_inc
        n_tmp = n[k] + dt05 * n_inc
        s_tmp = s[k] + dt05 * s_inc
        q_tmp = q[k] + dt05 * q_inc

        alpha_h_vtmp = 0.35 * math.exp(-(v_tmp + 58) / 20)
        alpha_n_vtmp = 0.05 * (v_tmp + 34) / (1 - math.exp(-0.1 * (v_tmp + 34)))
        beta_h_vtmp = 5.0 / (math.exp(-0.1 * (v_tmp + 28)) + 1)
        beta_n_vtmp = 0.625 * math.exp(-(v_tmp + 44) / 80)

        v_inc = (g_k * math.pow(n_tmp, 4.0) * (v_k - v_tmp) + g_na * math.pow(m_tmp, 3.0) * h_tmp * (v_na - v_tmp)
                 + g_l * (v_l - v_tmp) - g_syn * s_tmp * v_tmp + i_ext) / c
        h_inc = alpha_h_vtmp * (1 - h_tmp) - beta_h_vtmp * h_tmp
        n_inc = alpha_n_vtmp * (1 - n_tmp) - beta_n_vtmp * n_tmp
        s_inc = q_tmp * (1 - s_tmp) / tau_r - s_tmp / tau_d
        q_inc = -q_tmp / tau_dq

        v[k + 1] = v[k] + dt * v_inc
        alpha_m_v1 = 0.1 * (v[k + 1] + 35) / (1 - math.exp(-(v[k + 1] + 35) / 10))
        beta_m_v1 = 4 * math.exp(-(v[k + 1] + 60) / 18)
        m[k + 1] = alpha_m_v1 / (alpha_m_v1 + beta_m_v1)
        h[k + 1] = h[k] + dt * h_inc
        n[k + 1] = n[k] + dt * n_inc
        s[k + 1] = s[k] + dt * s_inc
        q[k + 1] = q[k] + dt * q_inc

    return v


def simulate_wb_entrainment_trace(g_syn, t_final, T=50.0, tau_d=2.0, tau_r=0.5, dt=0.01,
                                   c=1.0, g_k=9.0, g_na=35.0, g_l=0.1,
                                   v_k=-90.0, v_na=55.0, v_l=-65.0, i_ext=0.0):
    dt05 = dt / 2
    m_steps = round(t_final / dt)
    tau_dq = tau_d_q_function(tau_d, tau_r, tau_r)
    return _wb_entrainment_trace(g_syn, T, tau_d, tau_r, tau_dq, dt, dt05, m_steps,
                                  c, g_k, g_na, g_l, v_k, v_na, v_l, i_ext)


def spike_times_from_trace(v, t_final, dt=0.01):
    m_steps = round(t_final / dt)
    t = np.arange(m_steps + 1) * dt
    ind = np.where((v[:-1] >= -20) & (v[1:] < -20))[0]
    return (t[ind] * (-v[ind + 1] - 20) + t[ind + 1] * (20 + v[ind])) / (v[ind] - v[ind + 1])


@njit
def _wb_entrainment_spike_times(g_syn, T, tau_d, tau_r, tau_dq, dt, dt05, m_steps,
                                 c, g_k, g_na, g_l, v_k, v_na, v_l, i_ext, t_final):
    v = -65.0
    alpha_m0 = 0.1 * (v + 35) / (1 - math.exp(-(v + 35) / 10))
    beta_m0 = 4 * math.exp(-(v + 60) / 18)
    m = alpha_m0 / (alpha_m0 + beta_m0)
    alpha_h0 = 0.35 * math.exp(-(v + 58) / 20)
    beta_h0 = 5.0 / (math.exp(-0.1 * (v + 28)) + 1)
    h = alpha_h0 / (alpha_h0 + beta_h0)
    alpha_n0 = 0.05 * (v + 34) / (1 - math.exp(-0.1 * (v + 34)))
    beta_n0 = 0.625 * math.exp(-(v + 44) / 80)
    n = alpha_n0 / (alpha_n0 + beta_n0)
    q = 0.0
    s = 0.0

    t_spikes = np.empty(m_steps, dtype=np.float64)
    n_spikes = 0

    for k in range(m_steps):
        t = k * dt
        if k > 0 and abs(round(t / T) - t / T) < 1e-12:
            q = 1.0

        alpha_h_v = 0.35 * math.exp(-(v + 58) / 20)
        alpha_n_v = 0.05 * (v + 34) / (1 - math.exp(-0.1 * (v + 34)))
        beta_h_v = 5.0 / (math.exp(-0.1 * (v + 28)) + 1)
        beta_n_v = 0.625 * math.exp(-(v + 44) / 80)

        v_inc = (g_k * math.pow(n, 4.0) * (v_k - v) + g_na * math.pow(m, 3.0) * h * (v_na - v)
                 + g_l * (v_l - v) - g_syn * s * v + i_ext) / c
        h_inc = alpha_h_v * (1 - h) - beta_h_v * h
        n_inc = alpha_n_v * (1 - n) - beta_n_v * n
        s_inc = q * (1 - s) / tau_r - s / tau_d
        q_inc = -q / tau_dq

        v_tmp = v + dt05 * v_inc
        alpha_m_vtmp = 0.1 * (v_tmp + 35) / (1 - math.exp(-(v_tmp + 35) / 10))
        beta_m_vtmp = 4 * math.exp(-(v_tmp + 60) / 18)
        m_tmp = alpha_m_vtmp / (alpha_m_vtmp + beta_m_vtmp)
        h_tmp = h + dt05 * h_inc
        n_tmp = n + dt05 * n_inc
        s_tmp = s + dt05 * s_inc
        q_tmp = q + dt05 * q_inc

        alpha_h_vtmp = 0.35 * math.exp(-(v_tmp + 58) / 20)
        alpha_n_vtmp = 0.05 * (v_tmp + 34) / (1 - math.exp(-0.1 * (v_tmp + 34)))
        beta_h_vtmp = 5.0 / (math.exp(-0.1 * (v_tmp + 28)) + 1)
        beta_n_vtmp = 0.625 * math.exp(-(v_tmp + 44) / 80)

        v_inc = (g_k * math.pow(n_tmp, 4.0) * (v_k - v_tmp) + g_na * math.pow(m_tmp, 3.0) * h_tmp * (v_na - v_tmp)
                 + g_l * (v_l - v_tmp) - g_syn * s_tmp * v_tmp + i_ext) / c
        h_inc = alpha_h_vtmp * (1 - h_tmp) - beta_h_vtmp * h_tmp
        n_inc = alpha_n_vtmp * (1 - n_tmp) - beta_n_vtmp * n_tmp
        s_inc = q_tmp * (1 - s_tmp) / tau_r - s_tmp / tau_d
        q_inc = -q_tmp / tau_dq

        v_prev = v
        v = v + dt * v_inc
        alpha_m_v1 = 0.1 * (v + 35) / (1 - math.exp(-(v + 35) / 10))
        beta_m_v1 = 4 * math.exp(-(v + 60) / 18)
        m = alpha_m_v1 / (alpha_m_v1 + beta_m_v1)
        h = h + dt * h_inc
        n = n + dt * n_inc
        s = s + dt * s_inc
        q = q + dt * q_inc

        if v_prev >= -20 and v < -20 and t >= t_final / 2:
            t_next = (k + 1) * dt
            t_spike = (t * (-v - 20) + t_next * (20 + v_prev)) / (v_prev - v)
            t_spikes[n_spikes] = t_spike
            n_spikes += 1

    return t_spikes[:n_spikes]


def wb_entrainment_spike_times_for(g_syn, t_final, T=50.0, tau_d=2.0, tau_r=0.5, dt=0.01,
                                    c=1.0, g_k=9.0, g_na=35.0, g_l=0.1,
                                    v_k=-90.0, v_na=55.0, v_l=-65.0, i_ext=0.0):
    dt05 = dt / 2
    m_steps = round(t_final / dt)
    tau_dq = tau_d_q_function(tau_d, tau_r, tau_r)
    return _wb_entrainment_spike_times(g_syn, T, tau_d, tau_r, tau_dq, dt, dt05, m_steps,
                                        c, g_k, g_na, g_l, v_k, v_na, v_l, i_ext, t_final)

## WB Neuron: 1-to-1 Entrainment

A strong synapse ($\overline{g}_{\rm syn}=0.195$) locks one spike to
each pulse; a weaker one ($0.14$) still locks 1-to-1 but with a
different, still-repeatable phase $\delta$.

In [ ]:
def simulate_wb_neuron_entrained(t_final=800.0, T=50.0):
    v_strong = simulate_wb_entrainment_trace(0.195, t_final, T=T)
    v_weak = simulate_wb_entrainment_trace(0.14, t_final, T=T)
    t_spikes_strong = spike_times_from_trace(v_strong, t_final)
    delta = t_spikes_strong - np.floor(t_spikes_strong / T) * T
    return v_strong, v_weak, delta


def plot_wb_neuron_entrained(v_strong, v_weak, delta, t_final=800.0, T=50.0):
    t = np.arange(len(v_strong)) * 0.01
    period_lines = np.arange(1, round(t_final / T) + 2) * T

    fig, ax = plt.subplot_mosaic([['tl', 'tr'], ['bl', 'tr']], figsize=(11, 7))

    ax['tl'].plot(t, v_strong, '-k', linewidth=2)
    for tt in period_lines:
        ax['tl'].axvline(tt, color='r', linestyle=':', linewidth=1)
    ax['tl'].set_xlabel('$t$ [ms]')
    ax['tl'].set_ylabel('$v$ [mV]')
    ax['tl'].set_xlim(0, t_final)
    ax['tl'].set_ylim(-100, 50)
    ax['tl'].set_title(r'$\overline{g}_{\rm syn}=0.195$')

    ax['bl'].plot(np.arange(1, len(delta) + 1), delta, '.k', markersize=10)
    ax['bl'].set_xlabel('spike #')
    ax['bl'].set_ylabel(r'$\delta$ [ms]')
    ax['bl'].set_xlim(0, len(delta) + 1)
    ax['bl'].set_ylim(0, delta.max() + 1)

    ax['tr'].plot(t, v_weak, '-k', linewidth=2)
    for tt in period_lines:
        ax['tr'].axvline(tt, color='r', linestyle=':', linewidth=1)
    ax['tr'].set_xlabel('$t$ [ms]')
    ax['tr'].set_xlim(0, t_final)
    ax['tr'].set_ylim(-100, 50)
    ax['tr'].set_title(r'$\overline{g}_{\rm syn}=0.14$')

    plt.tight_layout()
    plt.show()

In [ ]:
plot_wb_neuron_entrained(*simulate_wb_neuron_entrained())

## WB Neuron: Irregular (Non-Locking) Response

At $\overline{g}_{\rm syn}=0.145$, the pulse-relative spike phase does
not settle into a repeating pattern, unlike the two panels above.

In [ ]:
def simulate_wb_neuron_irregular(T=50.0):
    panels = []
    for g_syn, t_final in [(0.180, 800.0), (0.145, 3200.0)]:
        v = simulate_wb_entrainment_trace(g_syn, t_final, T=T)
        t_spikes = spike_times_from_trace(v, t_final)
        delta = t_spikes - np.floor(t_spikes / T) * T
        panels.append((g_syn, t_final, v, delta))
    return panels


def plot_wb_neuron_panels(panels, T=50.0):
    fig, ax = plt.subplots(2, 2, figsize=(11, 7))
    for col, (g_syn, t_final, v, delta) in enumerate(panels):
        t = np.arange(len(v)) * 0.01
        period_lines = np.arange(1, round(t_final / T) + 2) * T

        ax[0, col].plot(t, v, '-k', linewidth=2)
        for tt in period_lines:
            ax[0, col].axvline(tt, color='r', linestyle=':', linewidth=1)
        ax[0, col].set_xlabel('$t$ [ms]')
        ax[0, col].set_xlim(0, t_final)
        ax[0, col].set_ylim(-100, 50)
        ax[0, col].set_title(rf'$\overline{{g}}_{{\rm syn}}={g_syn}$')

        ax[1, col].plot(np.arange(1, len(delta) + 1), delta, '.k', markersize=10)
        ax[1, col].set_xlabel('spike #')
        ax[1, col].set_xlim(0, len(delta) + 1)
        ax[1, col].set_ylim(0, delta.max() + 1)

    ax[0, 0].set_ylabel('$v$ [mV]')
    ax[1, 0].set_ylabel(r'$\delta$ [ms]')
    plt.tight_layout()
    plt.show()

In [ ]:
plot_wb_neuron_panels(simulate_wb_neuron_irregular())

## WB Neuron: n-to-1 Entrainment

At these synaptic strengths, several pulses occur per neuronal spike
rather than one per pulse -- count pulse intervals, not individual
pulses, to see the locking ratio.

In [ ]:
def simulate_wb_neuron_n_to_one(T=50.0):
    panels = []
    for g_syn, t_final in [(0.170, 800.0), (0.150, 1600.0)]:
        v = simulate_wb_entrainment_trace(g_syn, t_final, T=T)
        t_spikes = spike_times_from_trace(v, t_final)
        delta = t_spikes - np.floor(t_spikes / T) * T
        panels.append((g_syn, t_final, v, delta))
    return panels

In [ ]:
plot_wb_neuron_panels(simulate_wb_neuron_n_to_one())

## Entrainment Map over Synaptic Strength

Sweeps $\overline{g}_{\rm syn}$ over 201 values, measuring the mean
spike period (in units of $T$) and its coefficient of variation to mark
where the response settles into an exactly-repeating pattern (red).

In [ ]:
def simulate_wb_entrainment_intervals(a=0.142, b=0.195, N=200, T=50.0, t_final=10000.0):
    g_syn_vec = a + np.arange(N + 1) / N * (b - a)
    n_vec = np.zeros(len(g_syn_vec))
    sigma_vec = np.zeros(len(g_syn_vec))
    for ijk, g_syn in enumerate(g_syn_vec):
        t_spikes = wb_entrainment_spike_times_for(g_syn, t_final, T=T)
        periods = np.diff(t_spikes)
        sigma_vec[ijk] = (periods.max() - periods.min()) / periods.mean()
        n_vec[ijk] = periods.mean() / T
    return g_syn_vec, n_vec, sigma_vec


def plot_wb_entrainment_intervals(g_syn_vec, n_vec, sigma_vec):
    plt.figure(figsize=(6, 6))
    plt.plot(g_syn_vec, n_vec, '.k', markersize=5)
    ind = sigma_vec < 1e-3
    plt.plot(g_syn_vec[ind], n_vec[ind], '.r', markersize=10)

    plt.xlim(0.14, 0.19)
    plt.ylim(0, 18)
    plt.gca().set_box_aspect(1)
    plt.xlabel(r'$\overline{g}_{\rm syn}$')
    plt.ylabel('$n$')
    plt.xticks(np.arange(0.14, 0.191, 0.01))
    plt.yticks(range(5, 16, 5))
    plt.tight_layout()
    plt.show()

In [ ]:
# Numba-compiled, so this now takes well under a minute instead of the
# uncompiled sweep's roughly an hour.
plot_wb_entrainment_intervals(*simulate_wb_entrainment_intervals())